# LangSmith 시작하기: 설정과 개요

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. LangSmith가 무엇인지, 그리고 LangChain/LangGraph 생태계에서 어떤 역할을 하는지 설명할 수 있어요
2. LangSmith 계정을 만들고 API 키를 발급받을 수 있어요
3. `.env` 파일에 필요한 환경 변수(`LANGSMITH_TRACING`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT`)를 올바르게 설정할 수 있어요
4. LangSmith 프로젝트를 생성하고, 첫 번째 추적(Trace)이 UI에 기록되는 것을 확인할 수 있어요

## 사전 지식

- LangChain 기초: `ChatOpenAI`, `ChatPromptTemplate`, LCEL 파이프 연산자
- `.env` 파일과 환경 변수 개념
- `pip install` 패키지 설치 방법

## LangSmith란?

LangSmith는 LLM 애플리케이션을 **빌드, 디버그, 배포, 모니터링**하기 위한 관측성(Observability) 플랫폼이에요. LangChain/LangGraph와 긴밀하게 통합되어 있지만, 원칙적으로는 어떤 프레임워크와도 함께 사용할 수 있어요.

개발 과정에서 이런 의문이 생기지 않나요?

- "내 체인에서 어느 단계에서 시간이 오래 걸리지?"
- "검색 결과가 엉뚱하게 나왔는데 왜 그럴까?"
- "모델에 정확히 어떤 프롬프트가 들어갔지?"

LangSmith는 이런 질문에 답해주는 도구예요.

```mermaid
flowchart TD
    A[LangChain 애플리케이션<br/>체인, 에이전트, RAG]:::process --> B[LangSmith 자동 추적<br/>LANGSMITH_TRACING=true]
    B --> C{LangSmith 플랫폼}
    C --> D[Observability<br/>추적·디버깅]:::output
    C --> E[Evaluation<br/>품질 평가]:::output
    C --> F[Prompt Engineering<br/>프롬프트 관리]:::output
    C --> G[Deployment<br/>배포·모니터링]:::output
    D --> H[Trace 뷰어<br/>Run Tree 분석]:::storage
    E --> I[데이터셋<br/>평가 실험]:::storage
    F --> J[Prompt Hub<br/>버전 관리]:::storage

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

### LangSmith의 핵심 기능

| 기능 | 설명 |
|------|------|
| **Observability** | LLM 호출, 체인 실행의 전체 흐름을 시각적으로 확인해요 |
| **Evaluation** | 애플리케이션 품질을 데이터셋 기반으로 측정해요 |
| **Prompt Engineering** | 프롬프트를 버전 관리하고 A/B 테스트해요 |
| **Deployment** | 에이전트를 배포하고 실시간 모니터링해요 |

> 🔑 **핵심 개념**: LangSmith는 **프레임워크 독립적(Framework-agnostic)**이에요. LangChain 없이도 순수 OpenAI SDK, Anthropic SDK 등과 함께 사용할 수 있어요. 단, LangChain과 함께 쓸 때 가장 편리하게 자동 추적이 됩니다.

> 🎯 **강의 포인트**: LangSmith는 선택적 도구가 아니에요. 실무에서 LLM 앱을 운영하다 보면 "왜 이 응답이 나왔는지" 추적하는 일이 반드시 생겨요. 그때 LangSmith 없이는 블랙박스 안을 들여다볼 수 없습니다.

## 핵심 개념: Trace, Run, Project

LangSmith를 사용하기 전에 세 가지 개념을 이해해야 해요.

```mermaid
flowchart TD
    P[Project<br/>프로젝트]:::storage --> T1[Trace A<br/>전체 실행 한 번]:::process
    P --> T2[Trace B<br/>전체 실행 한 번]:::process
    T1 --> R1[Run: 체인<br/>run_type=chain]:::output
    R1 --> R2[Run: LLM 호출<br/>run_type=llm]:::output
    R1 --> R3[Run: 도구 호출<br/>run_type=tool]:::output
    R2 --> R4[Run: 임베딩<br/>run_type=embedding]:::output

    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

| 개념 | 설명 | 비유 |
|------|------|------|
| **Project** | Trace들의 모음. 기본값은 "default" | 폴더 |
| **Trace** | 단일 작업의 전체 실행 기록. Trace ID로 묶여요 | 작업 1건의 로그 파일 |
| **Run (Span)** | 하나의 작업 단위 (LLM 호출, 도구 실행 등) | 로그 파일 안의 한 줄 |

Run에는 다섯 가지 타입이 있어요:

| run_type | 설명 |
|----------|------|
| `llm` | LLM 호출 (토큰/비용 정보 포함) |
| `chain` | 여러 단계를 묶은 체인 실행 |
| `tool` | 도구/함수 실행 |
| `retriever` | 검색기 호출 |
| `embedding` | 임베딩 생성 |

## 1. 계정 생성 및 API Key 발급

LangSmith를 사용하려면 먼저 계정과 API 키가 필요해요.

### 계정 생성 절차

1. [smith.langchain.com](https://smith.langchain.com) 접속
2. 우측 상단 **Sign Up** 클릭
3. Google/GitHub 계정 또는 이메일로 가입
4. 워크스페이스(Workspace) 이름 설정 (조직 또는 개인 이름 사용 가능)

### API Key 발급 절차

1. 좌측 사이드바 **Settings** 클릭
2. **API Keys** 탭 선택
3. **Create API Key** 버튼 클릭
4. 키 유형 선택:
   - **Personal Access Token (PAT)**: 개인 사용, 본인 권한을 상속해요
   - **Service Key**: 프로덕션/팀 사용, 워크스페이스 수준 권한이에요
5. 생성된 키를 **즉시 복사**하세요 — 이후에는 다시 볼 수 없어요!

> ⚠️ **자주 하는 실수**: API 키는 생성 직후에만 전체 값을 볼 수 있어요. 창을 닫으면 다시 확인이 불가능합니다. 반드시 `.env` 파일에 즉시 저장하세요.

> 💡 **실무 팁**: 개인 학습에는 PAT를, 팀 프로젝트나 서버 배포에는 Service Key를 사용해요. Service Key는 특정 사람의 계정에 종속되지 않아서 팀원이 바뀌어도 키를 교체할 필요가 없어요.

## 2. SDK 설치

LangSmith를 사용하려면 필요한 패키지들을 설치해야 해요.

In [1]:
# ---------------------------------------------------
# 필요한 패키지 설치
# ---------------------------------------------------
# langsmith: LangSmith SDK (추적, 평가, 데이터셋 관리)
# langchain: LangChain 핵심 (자동 추적 지원)
# langchain-openai: OpenAI 모델 연동
# python-dotenv: .env 파일에서 환경 변수 로드
# !pip install -q -U langsmith langchain langchain-openai python-dotenv

## 3. 환경 변수 설정

LangSmith 추적을 활성화하려면 `.env` 파일에 환경 변수를 설정해야 해요.

프로젝트 루트에 `.env` 파일을 만들고 아래 내용을 추가하세요:

```
# OpenAI API 키
OPENAI_API_KEY=sk-...

# LangSmith 설정
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=lsv2_pt_...
LANGSMITH_PROJECT=LangSmith-Tutorial
```

### 환경 변수 설명

| 변수 | 필수 | 설명 |
|------|------|------|
| `LANGSMITH_TRACING` | 필수 | `true`로 설정하면 모든 LangChain 실행이 자동으로 추적돼요 |
| `LANGSMITH_API_KEY` | 필수 | smith.langchain.com에서 발급받은 API 키예요 |
| `LANGSMITH_PROJECT` | 선택 | 추적이 기록될 프로젝트 이름이에요. 기본값은 `"default"` |
| `OPENAI_API_KEY` | 필수 | OpenAI API 키예요 |

> 🔑 **핵심 개념**: `LANGSMITH_TRACING=true`만 설정하면 LangChain 코드에 **단 한 줄도 추가하지 않고** 자동으로 모든 LLM 호출이 추적돼요. 이것이 LangSmith의 가장 강력한 기능이에요.

> ⚠️ **자주 하는 실수**: 이전 버전 튜토리얼에서는 `LANGCHAIN_TRACING_V2`라는 변수명을 사용했어요. 현재 공식 문서 기준으로는 `LANGSMITH_TRACING`을 사용해야 해요. 두 변수 모두 작동하지만, 최신 방식을 따르는 것이 좋아요.

In [ ]:
# ---------------------------------------------------
# 환경 변수 로드 및 설정 확인
# ---------------------------------------------------
import os
from dotenv import load_dotenv

# .env 파일에서 환경 변수를 로드해요
load_dotenv()

# 이 노트북 전용 프로젝트
os.environ["LANGSMITH_PROJECT"] = "Ch01-Setup-Overview"

# LangSmith 관련 환경 변수가 올바르게 설정됐는지 확인해요
# API 키는 보안을 위해 앞 10자리만 출력해요
print("=== LangSmith 환경 변수 확인 ===")
print(f"LANGSMITH_TRACING    : {os.getenv('LANGSMITH_TRACING', '')}")
print(f"프로젝트              : {os.environ['LANGSMITH_PROJECT']}")
print(f"LANGSMITH_API_KEY    : {os.getenv('LANGSMITH_API_KEY', '')[:10]}...")
print(f"OPENAI_API_KEY       : {os.getenv('OPENAI_API_KEY', '')[:8]}...")

## 4. LangSmith Client와 프로젝트 관리

`langsmith.Client`를 사용하면 Python 코드에서 직접 프로젝트를 관리하고 데이터를 조회할 수 있어요.

In [3]:
# ---------------------------------------------------
# LangSmith Client 초기화
# ---------------------------------------------------
# langsmith.Client: LangSmith API와 통신하는 클라이언트
# 환경 변수 LANGSMITH_API_KEY를 자동으로 읽어요
from langsmith import Client

# Client 인스턴스 생성 — 연결이 정상적이면 에러 없이 완료돼요
client = Client()
print("LangSmith Client 초기화 완료!")

LangSmith Client 초기화 완료!


In [4]:
# ---------------------------------------------------
# 현재 워크스페이스의 프로젝트 목록 조회
# ---------------------------------------------------
# list_projects(): 워크스페이스의 모든 프로젝트를 반환해요
# 처음 사용 시 "default" 프로젝트만 있을 수 있어요
projects = list(client.list_projects())
print(f"현재 프로젝트 수: {len(projects)}")
for project in projects:
    print(f"  - {project.name} (ID: {str(project.id)[:8]}...)")

현재 프로젝트 수: 26
  - rag-transformer-qa-a06b9e42 (ID: 62747335...)
  - rag-embedding-similarity-e02fb6c6 (ID: 99e121ae...)
  - groundedness-combined-eacc9211 (ID: 4a5d6be8...)
  - groundedness-eval-9ef31f2c (ID: aaec23fa...)
  - rag-transformer-qa-ea128c11 (ID: 32a2d344...)
  - rag-embedding-similarity-da411902 (ID: 04ca2b24...)
  - groundedness-summary-38e1f1c4 (ID: ae6499e3...)
  - groundedness-eval-c94e70ef (ID: dcdc29d1...)
  - rag-transformer-qa-b11fac43 (ID: 858feca1...)
  - rag-embedding-similarity-138cb00f (ID: 458fce8b...)
  - rag-transformer-qa-4100a691 (ID: 8a765e59...)
  - rag-embedding-similarity-ed2aa2c4 (ID: 2f113bca...)
  - groundedness-eval-111c931c (ID: 105b0507...)
  - rag-transformer-qa-f63bd7f3 (ID: ee3dd654...)
  - groundedness-eval-54987fa1 (ID: c9163112...)
  - rag-embedding-similarity-5fdc2e53 (ID: 5f7becf0...)
  - rag-transformer-qa-2905a5ab (ID: 37100e43...)
  - groundedness-eval-dfeddc70 (ID: cb26318a...)
  - rag-embedding-similarity-51fefcb4 (ID: 9925a8f6...)


## 5. 첫 번째 자동 추적 예제

이제 LangSmith가 어떻게 자동으로 추적하는지 직접 확인해볼게요.

**`LANGSMITH_TRACING=true`만 설정하면 LangChain 코드를 수정하지 않아도 자동으로 추적돼요.** 아래 코드를 실행하고 LangSmith UI에서 결과를 확인해봅시다!

> 🎯 **강의 포인트**: 이 섹션이 핵심이에요. 학생들이 코드를 한 줄도 추가하지 않았는데 LangSmith UI에 Trace가 생겨나는 것을 직접 보면 "이게 어떻게 가능하지?"라는 반응이 나와요. LangChain은 내부적으로 Callback 시스템을 통해 자동으로 LangSmith에 데이터를 전송해요.

In [5]:
# ---------------------------------------------------
# 첫 번째 자동 추적 예제
# ---------------------------------------------------
# LANGSMITH_TRACING=true 설정만으로 아래 코드가 자동으로 추적돼요
# 코드 수정 없이 LangSmith UI에서 전체 실행 흐름을 볼 수 있어요!
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 모델 초기화 (gpt-4o-mini: 비용 효율적인 기본 모델)
model = ChatOpenAI(model="gpt-4o-mini")

# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_template(
    "다음 주제에 대해 한 문장으로 요약해줘: {topic}"
)

# 출력 파서 (AIMessage → 문자열 변환)
output_parser = StrOutputParser()

# LCEL 파이프 연산자로 체인 구성
chain = prompt | model | output_parser

# 체인 실행 — 이 한 줄이 LangSmith에 자동으로 추적돼요!
result = chain.invoke({"topic": "LangSmith"})
print(f"결과: {result}")

결과: LangSmith는 언어 모델 개발과 관련된 도구 및 리소스를 제공하여 연구자와 개발자들이 자연어 처리 애플리케이션을 효율적으로 구축할 수 있도록 돕는 플랫폼입니다.


위 코드를 실행한 후 [smith.langchain.com](https://smith.langchain.com)에 접속해서 확인해봐요:

1. 좌측 사이드바에서 **Projects** 클릭
2. `.env`에 설정한 프로젝트(또는 `default`)를 클릭
3. 방금 실행한 Trace를 클릭하면 **Run Tree**를 볼 수 있어요

Run Tree에서 볼 수 있는 것:
- `RunnableSequence` (체인 전체)
  - `ChatPromptTemplate` (프롬프트 포맷팅)
  - `ChatOpenAI` (LLM 호출 — 토큰 수, 소요 시간 포함)
  - `StrOutputParser` (결과 파싱)

> 💡 **실무 팁**: 프로젝트 이름을 의미 있게 설정하면 나중에 Trace를 찾기 쉬워요. 예를 들어 `customer-support-bot-dev`, `rag-pipeline-v2` 같은 이름으로 관리하면 여러 프로젝트를 동시에 운영할 때 유용해요.

In [6]:
# ---------------------------------------------------
# 프로젝트별로 Trace를 분리하는 방법
# ---------------------------------------------------
# ============================================================
# TODO: project_name을 다른 이름으로 변경해서 별도 프로젝트에 추적해봐요
# 힌트: config 딕셔너리의 "metadata"에 추가 정보를 붙일 수도 있어요
# 예상 결과: LangSmith UI에서 새 프로젝트 이름으로 Trace가 생겨요
# ============================================================

# config를 사용해 실행 시 프로젝트를 지정할 수 있어요
result2 = chain.invoke(
    {"topic": "Python 프로그래밍"},
    config={
        "run_name": "my-first-trace",          # Trace 이름 (UI에서 표시)
        "tags": ["tutorial", "part1"],         # 태그 (필터링에 사용)
        "metadata": {                           # 추가 메타데이터
            "student": "example-user",
            "session": "notebook-01"
        }
    }
)
print(f"결과: {result2}")
print("\nLangSmith UI에서 'tutorial' 태그로 이 Trace를 찾아보세요!")

결과: Python 프로그래밍은 간결하고 읽기 쉬운 문법을 통해 다양한 분야에서 소프트웨어 개발, 데이터 분석, 인공지능 등 여러 작업을 수행할 수 있게 하는 고급 프로그래밍 언어입니다.

LangSmith UI에서 'tutorial' 태그로 이 Trace를 찾아보세요!


In [7]:
# ---------------------------------------------------
# 노트북에서 Trace 제출 보장하기
# ---------------------------------------------------
# Jupyter 노트북은 비동기로 Trace를 전송해요
# 노트북이 종료되기 전에 모든 Trace가 전송됐는지 보장하려면
# wait_for_all_tracers()를 호출해야 해요
from langchain_core.tracers.langchain import wait_for_all_tracers

# 모든 대기 중인 Trace가 서버로 전송될 때까지 기다려요
wait_for_all_tracers()
print("모든 Trace가 LangSmith 서버로 전송됐어요!")

모든 Trace가 LangSmith 서버로 전송됐어요!


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **LangSmith**: LLM 애플리케이션의 관측성, 평가, 프롬프트 관리, 배포를 위한 플랫폼이에요
- **Trace / Run / Project**: Trace는 전체 작업 실행 기록, Run은 개별 단계, Project는 Trace들의 모음이에요
- **자동 추적**: `LANGSMITH_TRACING=true` 환경 변수만 설정하면 LangChain 코드 수정 없이 자동으로 추적돼요
- **API Key 발급**: smith.langchain.com > Settings > API Keys에서 발급하고, 생성 즉시 저장해야 해요
- **wait_for_all_tracers()**: 노트북에서 모든 Trace가 전송됐는지 보장하는 함수예요

## 다음 노트북 예고

다음 `02-First-Trace.ipynb`에서는 **`@traceable` 데코레이터**를 사용해 LangChain 외부의 일반 Python 함수도 추적하는 방법을 배워요. 중첩된 함수 호출로 부모-자식 Run 계층 구조를 만들고, LangSmith UI에서 Trace를 분석하는 방법도 알아볼게요.